# S3·MAR — Trazas ReAct: leerlas, protegerlas, mejorarlas

**MMIA 6013 · Semana 3 · Martes**

1. Ejecutar un ReAct simulado y leer sus trazas
2. Implementar un detector de loops
3. Implementar presupuesto de pasos
4. Mini-Reflexion: reintentar con la crítica del verificador

Todo local y determinista — el foco es la *mecánica* de los patrones.

In [ ]:
import json
import re

## Parte 1 — Tres trazas ReAct para diagnosticar

Cada traza es una lista de eventos Thought / Action / Observation / Answer.

In [ ]:
TRAZA_SANA = [
    {"tipo": "thought", "texto": "Necesito saber quién fundó SpaceX y en qué año."},
    {"tipo": "action", "tool": "buscar", "input": "fundación SpaceX año fundador"},
    {"tipo": "observation", "texto": "SpaceX fue fundada en 2002 por Elon Musk."},
    {"tipo": "thought", "texto": "Ahora necesito el año de nacimiento de Musk."},
    {"tipo": "action", "tool": "buscar", "input": "Elon Musk año de nacimiento"},
    {"tipo": "observation", "texto": "Elon Musk nació el 28 de junio de 1971."},
    {"tipo": "thought", "texto": "2002 - 1971 = 31 (o 30 antes de junio). Puedo responder."},
    {"tipo": "answer", "texto": "Tenía alrededor de 30-31 años."},
]

TRAZA_LOOP = [
    {"tipo": "thought", "texto": "Debo encontrar el precio de la acción de NimbusSoft."},
    {"tipo": "action", "tool": "buscar", "input": "precio acción NimbusSoft"},
    {"tipo": "observation", "texto": "Sin resultados."},
    {"tipo": "thought", "texto": "Intentaré con otras palabras."},
    {"tipo": "action", "tool": "buscar", "input": "NimbusSoft precio acción"},
    {"tipo": "observation", "texto": "Sin resultados."},
    {"tipo": "thought", "texto": "Probaré de nuevo."},
    {"tipo": "action", "tool": "buscar", "input": "precio acción NimbusSoft"},
    {"tipo": "observation", "texto": "Sin resultados."},
    {"tipo": "thought", "texto": "Intentaré con otras palabras."},
    {"tipo": "action", "tool": "buscar", "input": "NimbusSoft precio acción"},
    {"tipo": "observation", "texto": "Sin resultados."},
]

TRAZA_FABRICA = [
    {"tipo": "thought", "texto": "Necesito los ingresos de NimbusSoft en 2025."},
    {"tipo": "action", "tool": "buscar", "input": "ingresos NimbusSoft 2025"},
    {"tipo": "observation", "texto": "NimbusSoft es una empresa de software fundada en 2019."},
    {"tipo": "thought", "texto": "Según los resultados, los ingresos fueron 45 millones."},  # ← ¡nadie dijo eso!
    {"tipo": "answer", "texto": "Los ingresos de NimbusSoft en 2025 fueron de 45 millones USD."},
]

def imprimir_traza(traza):
    for e in traza:
        if e["tipo"] == "action":
            print(f"  Action:      {e['tool']}({e['input']!r})")
        else:
            print(f"  {e['tipo'].capitalize():12s} {e['texto']}")

print("=== TRAZA SANA ==="); imprimir_traza(TRAZA_SANA)

### ✏️ Ejercicio 1.1 (diagnóstico manual)
Imprime las otras dos trazas y anota en el comentario: ¿dónde exactamente se
rompe cada una? ¿Qué protección la habría atrapado?

In [ ]:
# imprimir_traza(TRAZA_LOOP)
# imprimir_traza(TRAZA_FABRICA)
#
# Diagnóstico TRAZA_LOOP:
# Diagnóstico TRAZA_FABRICA:

## Parte 2 — Detector de loops

### ✏️ Ejercicio 2.1
Implementa `detectar_loop(traza, ventana=6)` que devuelva True si hay
acciones repetidas (misma tool + input equivalente) dentro de la ventana.
Normaliza el input: minúsculas y orden alfabético de palabras — así
"precio acción NimbusSoft" ≡ "NimbusSoft precio acción".

In [ ]:
# Ejercicio 2.1 — repeticiones bajo normalización

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

**Limitación honesta:** la normalización por palabras no atrapa paráfrasis
("cotización de NimbusSoft"). En producción se combina con: límite de pasos,
límite de llamadas a la MISMA tool, y similitud de embeddings entre acciones.

## Parte 3 — Presupuesto: el seguro que nunca falla

### ✏️ Ejercicio 3.1
Envuelve un agente con **presupuesto de pasos** y con detección de repetición: si agota el
presupuesto o repite la misma llamada, se detiene y lo dice en la traza.

No es una precaución teórica. En la sesión 11, con la H200 de verdad, el agente pidió
`estadisticas_ventas` con `2025-03`, luego con `2024-03`, y siguió probando años porque el
error de la herramienta no decía qué meses existían: **el presupuesto fue lo único que lo
detuvo**. Un agente sin tope no falla, se queda dando vueltas y gastando.

In [ ]:
# Ejercicio 3.1 — envuelve un agente con presupuesto de pasos y de costo.
# Simulamos un agente que "quiere" dar 12 pasos; tu envoltura debe cortarlo.

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

## Parte 4 — Mini-Reflexion con verificador

La tarea: producir un JSON con formato exacto. El "verificador" es código
determinista (como los tests de un code agent). Si falla, la crítica entra al
contexto y el agente reintenta.

In [ ]:
def verificador(respuesta: str):
    """Devuelve (ok, critica)."""
    try:
        datos = json.loads(respuesta)
    except json.JSONDecodeError as e:
        return False, f"JSON inválido: {e}"
    if set(datos.keys()) != {"categoria", "urgencia"}:
        return False, f"Claves incorrectas: {sorted(datos.keys())} (esperaba categoria, urgencia)"
    if datos["categoria"] not in {"tecnico", "ventas"}:
        return False, f"categoria fuera de dominio: {datos['categoria']!r}"
    if not isinstance(datos["urgencia"], int):
        return False, "urgencia debe ser int"
    return True, "OK"

# Simulamos un modelo que mejora con la crítica (intento k usa las críticas previas)
INTENTOS_SIMULADOS = [
    'La categoría es técnica con urgencia 4: {"categoria": "tecnico", "urgencia": 4}',  # texto extra
    '{"categoria": "técnico", "urgencia": 4}',                                          # tilde: fuera de dominio
    '{"categoria": "tecnico", "urgencia": 4}',                                          # ✓
]

## Parte 4 — Reflexion: reintentar leyendo la crítica

### ✏️ Ejercicio 4.1
Implementa `reflexion(max_intentos=3)`: genera una salida, **verifícala**, y si falla,
vuelve a generar pasándole la crítica de lo que falló. Devuelve la primera salida que pase
y la traza de los intentos.

Lo que hay que mirar al correrlo no es que acabe pasando: es **de qué depende que pase**.
Reflexion solo funciona si el verificador dice algo accionable. «Está mal» no mejora nada;
«falta el campo `total`» sí. Es la misma lección que el martes de la semana 2 con los
mensajes de error de una herramienta, y la misma que dejó la sesión 11 al ver al modelo
adivinar años porque el error no decía qué meses existían.

In [ ]:
# Ejercicio 4.1 — implementa reflexion(max_intentos=3):
# en cada intento toma la salida simulada, verifica, y si falla registra la
# crítica (que un modelo real recibiría en su contexto). Devuelve la traza.

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

El patrón completo: **generar → verificar → criticar → regenerar con la crítica**.
Funciona porque el verificador aporta información que el modelo no tenía.
Sin verificador confiable, Reflexion degenera en el modelo dándose la razón.

In [ ]:
# 🤔 Reflexión final (comentario):
# Para el agente de TU proyecto (Lab 03):
# a) ¿Qué patrón usarás como baseline y por qué? (ReAct simple suele bastar)
# b) ¿Qué señal verificable podrías darle a un paso de Reflexion, si aplicara?
#
# a)
# b)

## Parte 5 — Leer una lista de patrones con criterio

Circula mucho contenido del tipo "los N patrones de agentes que todo ingeniero
debe conocer". Suelen ser útiles **y** estar mal ordenados: mezclan cosas que
viven en niveles distintos. Saber separarlas es la habilidad de hoy.

Una lista publicada recientemente enumeraba estos diez:

`ReAct` · `Chain of Thought` · `Parallel Execution` · `Self-Critique` ·
`Agent Handoff` · `Function Calling` · `LLM Workflow` · `Task Decomposition` ·
`Episodic Memory` · `Self-Reflection`

In [ ]:
# ✏️ Ejercicio 5.1 — clasifica los diez (comentario)
#
# Reparte cada elemento en UNA cubeta:
#
#   (1) PATRÓN DE CONTROL   — decide qué hace el agente y en qué orden
#   (2) TÉCNICA DE PROMPTING— cambia cómo se le pide al modelo, no el control
#   (3) MECANISMO           — la plomería que ejecuta, no una decisión de diseño
#   (4) TIPO DE MEMORIA     — qué se recuerda, no cómo se decide
#
# 1) patrón de control:
# 2) técnica de prompting:
# 3) mecanismo:
# 4) tipo de memoria:
#
# b) Dos de los diez son el MISMO patrón con dos nombres. ¿Cuáles?
#
# c) El autor de la lista vende un producto de memoria para agentes, y
#    "Episodic Memory" aparece en su lista. ¿Cambia eso lo que la lista vale?
#    ¿Cómo lo verificarías?
#

## Cierre

| Patrón | Compra | Cuesta | Protección asociada |
|---|---|---|---|
| ReAct | exploración anclada en hechos | 1 llamada/paso | detector de loops + presupuesto |
| Plan-Execute | coherencia y revisión del plan | rigidez | replanning |
| Reflexion | mejora con feedback | ×2–4 llamadas | verificador REAL |
| Task Decomposition | subproblemas manejables | coordinación y estado | límite de profundidad |
| Parallel Execution | latencia | costo simultáneo, orden no garantizado | presupuesto global |
| Agent Handoff | especialización | pérdida de contexto en el traspaso | contrato de traspaso explícito |

Chain of Thought, function calling y memoria episódica **no** entran en esta
tabla: no son patrones de control. Ver Parte 5.

**Mañana:** cuándo graduarte del loop artesanal a un grafo orquestado
(LangGraph) y qué es MCP.